In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']

from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.append('../')

import json
import pandas as pd
from glob import glob
from os import path

from utils.analyse_results import *
from utils.evaluation_framework import *

# Privacy Gain (MovieLens)

Visualisation of privacy gain against linkage attacks.  
Since utility evaluation is not yet implemented, only privacy gain is analysed.

## Load data

`load_results_linkage` extracts the dataset name from the portion after the last `_` in the filename,  
so for MovieLens files like `ml_col3_eff_rank_best` it would only get `best`.  
Here we override the Dataset column with the full name after `ResultsMIA_`.

In [ ]:
def load_ml_linkage_results(dirname, prefix='ml_col'):
    """Load ResultsMIA_ml_col*.json files for MovieLens and fix the Dataset name to the full name."""
    files = sorted(glob(path.join(dirname, f'ResultsMIA_{prefix}*.json')))
    
    all_results = []
    for fpath in files:
        dataset_name = path.basename(fpath).replace('ResultsMIA_', '').replace('.json', '')
        
        with open(fpath) as f:
            resDict = json.load(f)
        
        resList = []
        for tid, tres in resDict.items():
            for gm, gmDict in tres.items():
                for nr, nrDict in gmDict.items():
                    for fset, fsetDict in nrDict.items():
                        df = pd.DataFrame(fsetDict)
                        df['Run'] = nr
                        df['FeatureSet'] = fset
                        df['TargetModel'] = gm
                        df['TargetID'] = tid
                        df['Dataset'] = dataset_name
                        resList.append(df)
        
        results = pd.concat(resList)
        
        resAgg = []
        games = results.groupby(['Dataset', 'TargetID', 'TargetModel', 'FeatureSet', 'Run'])
        for gameParams, gameRes in games:
            tpSyn, fpSyn = get_tp_fp_rates(gameRes['AttackerGuess'], gameRes['Secret'])
            advantageSyn = get_mia_advantage(tpSyn, fpSyn)
            resAgg.append(gameParams + (tpSyn, fpSyn, advantageSyn, 1))
        
        resAgg = pd.DataFrame(resAgg,
            columns=['Dataset', 'TargetID', 'TargetModel', 'FeatureSet', 'Run',
                     'TPSyn', 'FPSyn', 'AdvantageSyn', 'AdvantageRaw'])
        resAgg['PrivacyGain'] = resAgg['AdvantageRaw'] - resAgg['AdvantageSyn']
        all_results.append(resAgg)
    
    if not all_results:
        raise FileNotFoundError(f'No ResultsMIA_{prefix}*.json files found in {dirname}')
    
    return pd.concat(all_results, ignore_index=True)

linkage = load_ml_linkage_results('../tests/linkage/')
print('Datasets:', sorted(linkage['Dataset'].unique()))
print('Models:', sorted(linkage['TargetModel'].unique()))
print(f'Total rows: {len(linkage)}')

## Overall summary

Mean privacy gain per Dataset × Model (aggregated over all FeatureSets, targets, and runs)

In [ ]:
summary = linkage.groupby(['Dataset', 'TargetModel'])['PrivacyGain'].agg(['mean', 'std', 'count'])
summary = summary.round(3)
summary

## Privacy gain distribution per FeatureSet

For each dataset, show a boxplot of privacy gain by model for each FeatureSet.

In [ ]:
datasets = sorted(linkage['Dataset'].unique())
feature_sets = sorted(linkage['FeatureSet'].unique())
models = sorted(linkage['TargetModel'].unique())

for ds in datasets:
    ds_data = linkage[linkage['Dataset'] == ds]
    
    fig, axes = plt.subplots(1, len(feature_sets), figsize=(7 * len(feature_sets), 6), sharey=True)
    if len(feature_sets) == 1:
        axes = [axes]
    
    for ax, fset in zip(axes, feature_sets):
        fset_data = ds_data[ds_data['FeatureSet'] == fset]
        sns.boxenplot(data=fset_data, x='TargetModel', y='PrivacyGain',
                      order=models, ax=ax)
        ax.set_title(f'{fset}', fontsize=FSIZELABELS)
        ax.set_xlabel('')
        ax.set_ylabel('$\\mathtt{PG}$' if ax == axes[0] else '', fontsize=FSIZELABELS)
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.tick_params(axis='y', labelsize=FSIZETICKS)
        ax.axhline(y=0, color='grey', linestyle='--', alpha=0.5)
        ax.set_ylim(-0.5, 1.5)
    
    fig.suptitle(f'Privacy Gain: {ds}', fontsize=FSIZELABELS, y=1.02)
    fig.tight_layout()
    plt.show()

## Per-target privacy gain (pointplot)

Using `plt_per_target_pg`, show per-target privacy gain by FeatureSet for each dataset.

In [ ]:
for ds in datasets:
    ds_data = linkage[linkage['Dataset'] == ds]
    ds_models = sorted(ds_data['TargetModel'].unique())
    
    for fset in feature_sets:
        print(f'--- {ds} / {fset} ---')
        fig = plt_per_target_pg(ds_data, ds_models, resFilter=('FeatureSet', fset))
        fig.suptitle(f'{ds}', fontsize=FSIZELABELS, y=1.35)
        plt.show()

## Generative vs Sanitisation comparison

Group models by category (Generative / Sanitisation) and compare privacy gain distributions.

In [ ]:
sanitisation_models = {'SanitiserNHS', 'SanitiserMondrian'}

def categorize_model(name):
    for s in sanitisation_models:
        if name.startswith(s):
            return 'Sanitisation'
    return 'Generative'

linkage['ModelCategory'] = linkage['TargetModel'].apply(categorize_model)

fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 6), sharey=True)
if len(datasets) == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets):
    ds_data = linkage[linkage['Dataset'] == ds]
    sns.boxenplot(data=ds_data, x='ModelCategory', y='PrivacyGain', ax=ax)
    ax.set_title(ds, fontsize=FSIZELABELS)
    ax.set_xlabel('')
    ax.set_ylabel('$\\mathtt{PG}$' if ax == axes[0] else '', fontsize=FSIZELABELS)
    ax.tick_params(axis='both', labelsize=FSIZETICKS)
    ax.axhline(y=0, color='grey', linestyle='--', alpha=0.5)

fig.suptitle('Generative vs Sanitisation', fontsize=FSIZELABELS, y=1.02)
fig.tight_layout()
plt.show()

## Cross-dataset comparison (when multiple results exist)

Compare col3 vs col19, best vs worst side by side. Skipped when only one result file is present.

In [ ]:
if len(datasets) > 1:
    # Compare only models common to all datasets
    common_models = set.intersection(*[
        set(linkage[linkage['Dataset'] == ds]['TargetModel'].unique())
        for ds in datasets
    ])
    common_models = sorted(common_models)
    
    cmp_data = linkage[linkage['TargetModel'].isin(common_models)]
    
    fig, ax = plt.subplots(figsize=(max(12, 3 * len(common_models)), 6))
    sns.boxenplot(data=cmp_data, x='TargetModel', y='PrivacyGain',
                  hue='Dataset', order=common_models, ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('$\\mathtt{PG}$', fontsize=FSIZELABELS)
    ax.tick_params(axis='x', rotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=FSIZETICKS)
    ax.axhline(y=0, color='grey', linestyle='--', alpha=0.5)
    ax.legend(title='Dataset', fontsize=12, title_fontsize=14)
    fig.suptitle('Privacy Gain: Dataset Comparison', fontsize=FSIZELABELS, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print(f'Only one dataset ({datasets[0]}) found; skipping cross-dataset comparison.')